# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/piasa19/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
**Method choice and why**

I chose **Random Forest** for this modeling lane because it can capture non-linear relationships between content freshness, CTR, search position, and impressions. It also provides feature importance, which helps explain which signals contribute most to the predictions.

I will compare the model with the Week-4 baseline using the same data and evaluation setup.


In [2]:
# ML-08 — Section 1: Method choice and why

import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Load the same dataset used in ML-07
data_path = "/content/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Signals used in the previous baseline
features = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

print("Dataset shape:", df.shape)
print("\nFeatures used:")
print(features)

print("\nFeature summary:")
display(df[features].describe())

Dataset shape: (30000, 44)

Features used:
['days_since_last_update', 'ctr', 'avg_position', 'impressions_90d']

Feature summary:


,days_since_last_update,ctr,avg_position,impressions_90d
count,30000.000000,30000.000000,30000.00000,30000.000000
mean,46.098300,0.510733,16.34238,5200.366300
std,42.078709,3.279162,15.21679,16838.019547
min,1.000000,0.000000,0.00000,1.000000
25%,20.000000,0.000000,6.20000,81.000000
50%,20.000000,0.070000,10.80000,731.000000
75%,104.000000,0.290000,22.30000,3615.250000
max,373.000000,100.000000,245.00000,517715.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [4]:
# ML-08 — Section 2: Split design

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Create the actual target
# 1 = declining, 0 = not declining
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Client-aware split
# Pages from the same client should not appear in both train and test.
clients = df["client_id"].fillna("unknown").astype(str).unique()

rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(clients)

test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = df["client_id"].fillna("unknown").astype(str).isin(test_clients)

train_df = df.loc[~test_mask].copy()
test_df = df.loc[test_mask].copy()

print("Split strategy: client holdout")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nTarget rate:")
print("Train:", round(train_df["is_declining_label"].mean(), 4))
print("Test :", round(test_df["is_declining_label"].mean(), 4))

# Safety check: no client overlap
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("\nClient overlap:", len(overlap))

assert len(overlap) == 0
assert train_df["is_declining_label"].nunique() == 2
assert test_df["is_declining_label"].nunique() == 2

print("Split check passed.")

Split strategy: client holdout
Training rows: 27675
Test rows: 2325
Training clients: 26
Test clients: 6

Target rate:
Train: 0.5548
Test : 0.391

Client overlap: 0
Split check passed.


**Split design**

I used a client-level holdout split with 80% of clients for training and 20% for testing. This is more honest than a random row split because pages from the same client should not appear in both train and test.

The target is `is_declining_label`, where `1` means the content is currently marked as declining by `trend_direction == "down"`. The test set is kept separate for evaluation.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# ML-08 — Section 3: Train + Compare vs Baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# ------------------------------------------------------------
# 1. Features
# ------------------------------------------------------------

features = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

X_train = train_df[features].copy()
y_train = train_df["is_declining_label"].copy()

X_test = test_df[features].copy()
y_test = test_df["is_declining_label"].copy()

# ------------------------------------------------------------
# 2. Train Random Forest
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Probability of being declining
model_prob = model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# 3. Create ML ranking
# ------------------------------------------------------------

model_results = test_df[
    ["content_id", "client_id", "is_declining_label"]
].copy()

model_results["model_score"] = model_prob

model_results = model_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Recreate Week-4 baseline on the TEST set
# ------------------------------------------------------------

baseline_test = test_df.copy()

# Expected CTR by position bucket
def position_bucket(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "positions_11_20"
    else:
        return "deep"

baseline_test["position_bucket"] = (
    baseline_test["avg_position"].apply(position_bucket)
)

expected_ctr = (
    train_df.assign(
        position_bucket=train_df["avg_position"].apply(position_bucket)
    )
    .groupby("position_bucket")["ctr"]
    .median()
)

baseline_test["expected_ctr"] = (
    baseline_test["position_bucket"].map(expected_ctr)
)

baseline_test["ctr_gap"] = (
    baseline_test["expected_ctr"] - baseline_test["ctr"]
).clip(lower=0)

# IMPORTANT:
# Rank using training distributions only.
staleness_reference = train_df["days_since_last_update"].rank(pct=True)
ctr_gap_reference = (
    train_df.assign(
        position_bucket=train_df["avg_position"].apply(position_bucket)
    )
    .assign(
        expected_ctr=lambda x:
        x["position_bucket"].map(expected_ctr)
    )
    .assign(
        ctr_gap=lambda x:
        (x["expected_ctr"] - x["ctr"]).clip(lower=0)
    )["ctr_gap"]
)

def percentile_score(value, reference):
    return (reference <= value).mean()

baseline_test["staleness_score"] = baseline_test[
    "days_since_last_update"
].apply(
    lambda x: percentile_score(x, train_df["days_since_last_update"])
)

baseline_test["ctr_opportunity_score"] = baseline_test[
    "ctr_gap"
].apply(
    lambda x: percentile_score(x, ctr_gap_reference)
)

baseline_test["baseline_score"] = (
    0.60 * baseline_test["staleness_score"]
    + 0.40 * baseline_test["ctr_opportunity_score"]
)

baseline_results = baseline_test[
    ["content_id", "client_id", "is_declining_label", "baseline_score"]
].copy()

baseline_results = baseline_results.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. Precision@50
# ------------------------------------------------------------

def precision_at_k(results, score_column, k=50):
    top_k = results.head(k)
    return top_k["is_declining_label"].mean()

baseline_p50 = precision_at_k(
    baseline_results,
    "baseline_score",
    50
)

model_p50 = precision_at_k(
    model_results,
    "model_score",
    50
)

# ------------------------------------------------------------
# 6. Model vs Baseline table
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

comparison["Precision@50"] = comparison["Precision@50"].round(4)

print("========== MODEL VS BASELINE ==========")
display(comparison)

# ------------------------------------------------------------
# 7. Feature importance
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========")
display(feature_importance)

========== MODEL VS BASELINE ==========


,Method,Precision@50
0,Week-4 Baseline,0.22
1,Random Forest,0.66



========== FEATURE IMPORTANCE ==========


,feature,importance
3,impressions_90d,0.389905
2,avg_position,0.310280
0,days_since_last_update,0.201896
1,ctr,0.097918


**Train + compare**

I trained a Random Forest classifier to predict the declining-content label. The model is evaluated on the held-out clients using Precision@50.

The Week-4 baseline is also evaluated on the same test clients and the same Precision@50 metric. This makes the comparison consistent and avoids evaluating the model and baseline on different data.

Feature importance is reported to understand which available signals contribute most to the model.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# ML-08 — Section 4: Errors and interpretation

# ------------------------------------------------------------
# 1. Add predictions to the model results
# ------------------------------------------------------------

model_results["predicted_label"] = (
    model_results["model_score"] >= 0.5
).astype(int)

# ------------------------------------------------------------
# 2. Find false positives and false negatives
# ------------------------------------------------------------

false_positives = model_results[
    (model_results["predicted_label"] == 1) &
    (model_results["is_declining_label"] == 0)
].copy()

false_negatives = model_results[
    (model_results["predicted_label"] == 0) &
    (model_results["is_declining_label"] == 1)
].copy()

print("========== ERROR SUMMARY ==========")
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# ------------------------------------------------------------
# 3. Show highest-confidence false positives
# ------------------------------------------------------------

print("\n========== TOP FALSE POSITIVES ==========")

display(
    false_positives
    .sort_values("model_score", ascending=False)
    .head(10)
)

# ------------------------------------------------------------
# 4. Show highest-confidence false negatives
# ------------------------------------------------------------

print("\n========== TOP FALSE NEGATIVES ==========")

display(
    false_negatives
    .sort_values("model_score", ascending=True)
    .head(10)
)

# ------------------------------------------------------------
# 5. Feature importance
# ------------------------------------------------------------

print("\n========== FEATURE INTERPRETATION ==========")

for _, row in feature_importance.iterrows():
    print(
        f"{row['feature']}: "
        f"{row['importance']:.4f}"
    )

# ------------------------------------------------------------
# 6. Simple interpretation
# ------------------------------------------------------------

top_feature = feature_importance.iloc[0]["feature"]

print("\n========== INTERPRETATION ==========")
print(
    f"The most influential feature in this Random Forest model "
    f"was '{top_feature}'."
)

print(
    "False positives are pages predicted as declining but not labelled "
    "as declining in the observed data."
)

print(
    "False negatives are pages labelled as declining but receiving "
    "a lower model score."
)

========== ERROR SUMMARY ==========
False positives: 503
False negatives: 288

========== TOP FALSE POSITIVES ==========


,content_id,client_id,is_declining_label,model_score,predicted_label
0,content_107776820988,client_d4735e3a26,0,0.795037,1
1,content_d7807f7dc347,client_d4735e3a26,0,0.780675,1
5,content_5ccc2e507f1f,client_d4735e3a26,0,0.716044,1
13,content_06f581dd9383,client_d4735e3a26,0,0.707528,1
19,content_51715e336a42,client_f74efabef1,0,0.702009,1
21,content_c099820b953c,client_f74efabef1,0,0.700813,1
23,content_a90410bf0556,client_f74efabef1,0,0.700681,1
25,content_9d931e9a2b19,client_f74efabef1,0,0.700407,1
26,content_b4feca62598e,client_f74efabef1,0,0.700242,1
29,content_f5013794ba57,client_f74efabef1,0,0.699818,1



========== TOP FALSE NEGATIVES ==========


,content_id,client_id,is_declining_label,model_score,predicted_label
2321,content_28b4223f4e5f,client_98a3ab7c34,1,0.000542,0
2071,content_34b14c00f80c,client_d4735e3a26,1,0.018332,0
2034,content_c0af3d6f9dd3,client_d4735e3a26,1,0.057007,0
1997,content_a8cee66e4788,client_d4735e3a26,1,0.076895,0
1957,content_86748254b6bf,client_d4735e3a26,1,0.092145,0
1949,content_472ce7ae14c0,client_d4735e3a26,1,0.094201,0
1937,content_943e6f110fdd,client_d4735e3a26,1,0.105115,0
1936,content_79ac977c6e0b,client_f74efabef1,1,0.106338,0
1935,content_8bc35200c219,client_d4735e3a26,1,0.106698,0
1929,content_453d445896e5,client_d4735e3a26,1,0.120154,0



========== FEATURE INTERPRETATION ==========
impressions_90d: 0.3899
avg_position: 0.3103
days_since_last_update: 0.2019
ctr: 0.0979

========== INTERPRETATION ==========
The most influential feature in this Random Forest model was 'impressions_90d'.
False positives are pages predicted as declining but not labelled as declining in the observed data.
False negatives are pages labelled as declining but receiving a lower model score.


**Errors and interpretation**

The error review separates false positives from false negatives. False positives are pages the model ranked as declining but were not labelled as declining, while false negatives are declining pages that received lower model scores.

The feature-importance results provide a simple interpretation of which observed signals the Random Forest relied on most. These results should be treated as model-specific importance rather than causal effects.

The error examples also show where the model can disagree with the observed label and where additional investigation may be useful.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.